# setup

In [ ]:
import os, scipy.io as sio, pandas as pd
import numpy as np 
import pymc as pm, arviz as az, nutpie
import matplotlib.pyplot as plt
import scipy.stats as st
import pytensor.tensor as pt

root_folder = '/home/habv0/Q5078local'
output_folder = './model-outputs'

In [ ]:
def load_data(file):
    assert os.path.isfile(file)
    mat = sio.loadmat(file)

    df = pd.DataFrame()
    for field in mat['df'].dtype.names:
        df[field] = mat['df'][field][0,0].flatten()
        
    return df

# load data
filename = os.path.join(root_folder,'contrLocalBadpt/derivatives/eeg/ic/compiled-data-struct.mat')
df = load_data(filename)
df['y'] = (df['choice']=='right').astype(int)
df.loc[df['contrast']==df['contrast'].min(),'contrast'] = 0

dat = df.copy()
dat = dat.loc[dat['blinks']==0,:]

# code unique sessions
dat['p'] = dat.groupby(['sub','ses']).ngroup()

# code the contrast intervals c1 = left interval contrast, c2 = right interval contrast
dat['c1'] = dat['contrast']
dat.loc[(dat['side'] == 'right'), 'c1'] = 0

dat['c2'] = dat['contrast']
dat.loc[(dat['side'] == 'left'), 'c2'] = 0

dat['C'] = dat['c2'] - dat['c1']

# subtract phase from ERP best predicted
dat['phi'] = dat['phi_echt']
dat['phi'] = np.arctan2(np.sin(dat['phi']-dat['phase_offset']), np.cos(dat['phi']-dat['phase_offset']))
dat.loc[np.isnan(dat['phi']),'phi'] = 0  # this will be nullified by the 0 amplitude here anyway

dat['pow'] = np.log1p(dat['power'])
dat.loc[np.isnan(dat['pow']),'pow'] = 0
dat['pow'] = dat.groupby('p')['pow'].transform(lambda x: x/x[x>0].mean())  # fix scale of non-zero
plt.hist(dat['pow'],50);

# fit behavioural model

In [ ]:
coords = dict()
coords['p'] = np.unique(dat['p'])
coords['param'] = ['n','c50','b0','eps']

with pm.Model(coords=coords) as model:
    c1 = pm.Data('c1', dat['c1'])
    c2 = pm.Data('c2', dat['c2'])
    y = pm.Data('y', dat['y'])
    p = pm.Data('p', dat['p'])

    # hierarchical parameters
    mu_n = pm.HalfNormal('mu_n', 3)  # slope
    mu_c50 = pm.Beta('mu_c50', 3, 2)
    mu_b0 = pm.Normal('mu_b0', 0, 1)
    mu_eps__ = pm.Beta('mu_eps__',1,5)
    mu_eps = pm.Deterministic('mu_eps', mu_eps__*.5)

    sigma = pm.HalfNormal('sigma', .5, dims='param')
    Z = pm.Normal('Z', dims=['param','p'])

    # participant parameters
    Rmax = 5
    n = pm.Deterministic('n', mu_n*np.exp(Z[0]*sigma[0]), dims='p')
    c50 = pm.Deterministic('c50', pm.math.invlogit(pm.math.logit(mu_c50)+Z[1]*sigma[1]), dims='p')
    b0 = pm.Deterministic('b0', mu_b0+Z[2]*sigma[2], dims='p')
    eps = pm.Deterministic('eps', .5*pm.math.invlogit(pm.math.logit(mu_eps__) + Z[3]*sigma[3]), dims='p')

    # transducer function
    T1 = c1**n[p] / (c1**n[p] + c50[p]**n[p])
    T2 = c2**n[p] / (c2**n[p] + c50[p]**n[p])

    d1 = Rmax*T1
    d2 = Rmax*T2
    psi = eps[p] + (1-2*eps[p])*(1-pm.math.invprobit((b0[p] - (d2-d1))/np.sqrt(2)))
    yobs = pm.Bernoulli('y_obs', p=psi, observed=y)

In [ ]:
compiled = nutpie.compile_pymc_model(model, backend='jax', gradient_backend='jax')
trace = nutpie.sample(compiled, chains=4)
with model:
    pm.compute_log_likelihood(trace)

In [ ]:
# save
az.to_netcdf(trace, os.path.join(output_folder,'inhib-model-2AFC-fit-beh.nc'))

# fit response gain

In [ ]:
coords = dict()
coords['p'] = np.unique(dat['p'])
coords['param'] = ['n','c50','b0','eps','g']

with pm.Model(coords=coords) as model:
    c1 = pm.Data('c1', dat['c1'])
    c2 = pm.Data('c2', dat['c2'])
    y = pm.Data('y', dat['y'])
    p = pm.Data('p', dat['p'])
    phase = pm.Data('phase', dat['phi'])
    power = pm.Data('power', dat['pow'])

    # hierarchical parameters
    mu_n = pm.HalfNormal('mu_n', 3)  # slope
    mu_c50 = pm.Beta('mu_c50', 3, 2)
    mu_b0 = pm.Normal('mu_b0', 0, 1)
    mu_eps__ = pm.Beta('mu_eps__',1,5)
    mu_eps = pm.Deterministic('mu_eps', mu_eps__*.5)
    mu_g = pm.Normal('mu_g', 0,2)  # inhibition of response gain

    sigma = pm.HalfNormal('sigma', .5, dims='param')
    Z = pm.Normal('Z', dims=['param','p'])

    # participant parameters
    Rmax = 5
    n = pm.Deterministic('n', mu_n*np.exp(Z[0]*sigma[0]), dims='p')
    c50 = pm.Deterministic('c50', pm.math.invlogit(pm.math.logit(mu_c50)+Z[1]*sigma[1]), dims='p')
    b0 = pm.Deterministic('b0', mu_b0+Z[2]*sigma[2], dims='p')
    eps = pm.Deterministic('eps', .5*pm.math.invlogit(pm.math.logit(mu_eps__) + Z[3]*sigma[3]), dims='p')
    g = pm.Deterministic('g', mu_g+Z[4]*sigma[4], dims='p')

    # transducer function
    T1 = c1**n[p] / (c1**n[p] + c50[p]**n[p])
    T2 = c2**n[p] / (c2**n[p] + c50[p]**n[p])

    # inhibition 
    m = pm.Beta('m', 1,1)  
    I = power*(1+m*np.cos(phase))

    d1 = Rmax*np.exp(g[p]*I)*T1
    d2 = Rmax*np.exp(g[p]*I)*T2
    psi = eps[p] + (1-2*eps[p])*(1-pm.math.invprobit((b0[p] - (d2-d1))/np.sqrt(2)))
    yobs = pm.Bernoulli('y_obs', p=psi, observed=y)

In [ ]:
compiled = nutpie.compile_pymc_model(model, backend='jax', gradient_backend='jax')
trace = nutpie.sample(compiled, chains=4, seed=12345)
with model:
    pm.compute_log_likelihood(trace)

In [ ]:
# save
az.to_netcdf(trace, os.path.join(output_folder,'inhib-model-2AFC-fit-g.nc'))

# fit baseline

In [ ]:

coords = dict()
coords['p'] = np.unique(dat['p'])
coords['param'] = ['n','c50','b0','eps','b']

with pm.Model(coords=coords) as model:
    c1 = pm.Data('c1', dat['c1'])
    c2 = pm.Data('c2', dat['c2'])
    y = pm.Data('y', dat['y'])
    p = pm.Data('p', dat['p'])
    phase = pm.Data('phase', dat['phi'])
    power = pm.Data('power', dat['pow'])

    # hierarchical parameters
    mu_n = pm.HalfNormal('mu_n', 3)  # slope
    mu_c50 = pm.Beta('mu_c50', 3, 2)
    mu_b0 = pm.Normal('mu_b0', 0, 1)
    mu_eps__ = pm.Beta('mu_eps__',1,5)
    mu_eps = pm.Deterministic('mu_eps', mu_eps__*.5)
    mu_b = pm.Normal('mu_b', 0, 2)  # inhibition of baseline

    sigma = pm.HalfNormal('sigma', .5, dims='param')
    Z = pm.Normal('Z', dims=['param','p'])

    # participant parameters
    Rmax = 5
    n = pm.Deterministic('n', mu_n*np.exp(Z[0]*sigma[0]), dims='p')
    c50 = pm.Deterministic('c50', pm.math.invlogit(pm.math.logit(mu_c50)+Z[1]*sigma[1]), dims='p')
    b0 = pm.Deterministic('b0', mu_b0+Z[2]*sigma[2], dims='p')
    eps = pm.Deterministic('eps', .5*pm.math.invlogit(pm.math.logit(mu_eps__) + Z[3]*sigma[3]), dims='p')
    b = pm.Deterministic('b', mu_b+Z[4]*sigma[4], dims='p')

    # transducer function
    T1 = c1**n[p] / (c1**n[p] + c50[p]**n[p])
    T2 = c2**n[p] / (c2**n[p] + c50[p]**n[p])

    # inhibition 
    m = pm.Beta('m', 1,1)  
    I = power*(1+m*np.cos(phase))

    d1 = Rmax*T1
    d2 = Rmax*T2
    psi = eps[p] + (1-2*eps[p])*(1-pm.math.invprobit(((b0[p]+b[p]*I) - (d2-d1))/np.sqrt(2)))
    yobs = pm.Bernoulli('y_obs', p=psi, observed=y)

In [ ]:
compiled = nutpie.compile_pymc_model(model, backend='jax', gradient_backend='jax')
trace = nutpie.sample(compiled, chains=4, seed=12345)
with model:
    pm.compute_log_likelihood(trace)

In [ ]:
# save
az.to_netcdf(trace, os.path.join(output_folder,'inhib-model-2AFC-fit-b.nc'))

# fit contrast gain model

In [ ]:
coords = dict()
coords['p'] = np.unique(dat['p'])
coords['param'] = ['n','c50','b0','eps','Cg']

with pm.Model(coords=coords) as model:
    c1 = pm.Data('c1', dat['c1'])
    c2 = pm.Data('c2', dat['c2'])
    y = pm.Data('y', dat['y'])
    p = pm.Data('p', dat['p'])
    phase = pm.Data('phase', dat['phi'])
    power = pm.Data('power', dat['pow'])

    # hierarchical parameters
    mu_n = pm.HalfNormal('mu_n', 3)  # slope
    mu_c50 = pm.Beta('mu_c50', 3, 2)
    mu_b0 = pm.Normal('mu_b0', 0, 1)
    mu_eps__ = pm.Beta('mu_eps__',1,5)
    mu_eps = pm.Deterministic('mu_eps', mu_eps__*.5)
    mu_Cg = pm.Normal('mu_Cg', 0, 2)  

    sigma = pm.HalfNormal('sigma', .5, dims='param')
    Z = pm.Normal('Z', dims=['param','p'])

    # participant parameters
    Rmax = 5
    n = pm.Deterministic('n', mu_n*np.exp(Z[0]*sigma[0]), dims='p')
    c50 = pm.Deterministic('c50', pm.math.invlogit(pm.math.logit(mu_c50)+Z[1]*sigma[1]), dims='p')
    b0 = pm.Deterministic('b0', mu_b0+Z[2]*sigma[2], dims='p')
    eps = pm.Deterministic('eps', .5*pm.math.invlogit(pm.math.logit(mu_eps__) + Z[3]*sigma[3]), dims='p')
    Cg = pm.Deterministic('Cg', mu_Cg+Z[4]*sigma[4], dims='p')

    # inhibition 
    m = pm.Beta('m', 1,1)  
    I = power*(1+m*np.cos(phase))

    # transducer function
    c50_ = pm.math.invlogit(pm.math.logit(c50[p]) - Cg[p]*I)
    T1 = c1**n[p] / (c1**n[p] + c50_**n[p])
    T2 = c2**n[p] / (c2**n[p] + c50_**n[p])

    d1 = Rmax*T1
    d2 = Rmax*T2
    psi = eps[p] + (1-2*eps[p])*(1-pm.math.invprobit((b0[p] - (d2-d1))/np.sqrt(2)))
    yobs = pm.Bernoulli('y_obs', p=psi, observed=y)

In [ ]:
compiled = nutpie.compile_pymc_model(model, backend='jax', gradient_backend='jax')
trace = nutpie.sample(compiled, chains=4, seed=12345)
with model:
    pm.compute_log_likelihood(trace)

In [ ]:
# save
az.to_netcdf(trace, os.path.join(output_folder,'inhib-model-2AFC-fit-Cg.nc'))

# fit noise model

In [ ]:
coords = dict()
coords['p'] = np.unique(dat['p'])
coords['param'] = ['n','c50','b0','eps','Nz']

with pm.Model(coords=coords) as model:
    c1 = pm.Data('c1', dat['c1'])
    c2 = pm.Data('c2', dat['c2'])
    y = pm.Data('y', dat['y'])
    p = pm.Data('p', dat['p'])
    phase = pm.Data('phase', dat['phi'])
    power = pm.Data('power', dat['pow'])

    # hierarchical parameters
    mu_n = pm.HalfNormal('mu_n', 3)  # slope
    mu_c50 = pm.Beta('mu_c50', 3, 2)
    mu_b0 = pm.Normal('mu_b0', 0, 1)
    mu_eps__ = pm.Beta('mu_eps__',1,5)
    mu_eps = pm.Deterministic('mu_eps', mu_eps__*.5)
    mu_Nz = pm.Normal('mu_Nz', 0, 2) 

    sigma = pm.HalfNormal('sigma', .5, dims='param')
    Z = pm.Normal('Z', dims=['param','p'])

    # participant parameters
    Rmax = 5
    n = pm.Deterministic('n', mu_n*np.exp(Z[0]*sigma[0]), dims='p')
    c50 = pm.Deterministic('c50', pm.math.invlogit(pm.math.logit(mu_c50)+Z[1]*sigma[1]), dims='p')
    b0 = pm.Deterministic('b0', mu_b0+Z[2]*sigma[2], dims='p')
    eps = pm.Deterministic('eps', .5*pm.math.invlogit(pm.math.logit(mu_eps__) + Z[3]*sigma[3]), dims='p')
    Nz = pm.Deterministic('Nz', mu_Nz+Z[4]*sigma[4], dims='p')

    # inhibition 
    m = pm.Beta('m', 1,1)  
    I = power*(1+m*np.cos(phase))

    # transducer function
    T1 = c1**n[p] / (c1**n[p] + c50[p]**n[p])
    T2 = c2**n[p] / (c2**n[p] + c50[p]**n[p])

    d1 = Rmax*T1
    d2 = Rmax*T2
    zprob = b0[p] - (d2-d1)
    zprob /= pm.math.exp(Nz[p]*I)
    psi = eps[p] + (1-2*eps[p])*(1-pm.math.invprobit(zprob / np.sqrt(2)))
    yobs = pm.Bernoulli('y_obs', p=psi, observed=y)

In [ ]:
compiled = nutpie.compile_pymc_model(model, backend='jax', gradient_backend='jax')
trace = nutpie.sample(compiled, chains=4, seed=12345)
with model:
    pm.compute_log_likelihood(trace)

In [ ]:
# save
az.to_netcdf(trace, os.path.join(output_folder,'inhib-model-2AFC-fit-Nz.nc'))